# V2 Phase 16 — Post-hoc LLM-as-judge faithfulness (Colab GPU)

**Before running:** Runtime → Change runtime type → **GPU** (T4 or better).

This notebook scores the **frozen Phase 15** 420-case JSONL. It does **not** rerun Single-Agent, Multi-Agent, or Multi-Agent + UQ. It does **not** regenerate answers or retrieve.

Metric: **LLM-as-judge faithfulness (Qwen3-8B, custom/RAGAS-inspired)**. **Not official RAGAS Faithfulness.**

Judge: Qwen3-8B Q4_K_M, `llama_cpp`, `n_ctx=4096`, `temperature=0.0`, `max_new_tokens=32`.

Input per saved case: question + `retrieved_evidence[].text` + claim (UQ uses `configuration.draft_answer`). Gold FinQA context and gold answers are **not** given to the judge.

Does **not** modify the frozen 140/40, T=0.65, Phase 15 JSONL, Phase 16 CPU metrics, or V1.

## Setup

Push latest V2 (`scripts/run_judge.py`) to branch `main`, then run **this notebook on Colab GPU**.

Historical Colab clones used a previous development workspace (legacy launch configuration).

Requires the frozen Phase 15 JSONL on Drive:
`MyDrive/MSc-RAG/results/raw/phase15_benchmark/phase15_20260826T203744Z_dae9c3a4/cases.jsonl`

SHA-256 must be `f5256ae40fa8db0d6172ff9f4083bbde6c1c4fdb47916baa73529bc8215caafa`.

**Outputs:** `results/raw/phase16_judge/{run_id}/judge.jsonl` and Drive copies under `MyDrive/MSc-RAG/results/raw/phase16_judge/`.

If Colab disconnects: run the **resume** cell. Do **not** start from case 1.

## 1. Clone GitHub repo and enter V2

In [ ]:
from pathlib import Path
import os
import platform
import subprocess
import sys

if platform.system() == 'Darwin' or not Path('/content').exists():
    raise RuntimeError('Open this notebook on Colab GPU. Do not run the 420-case judge on the Mac.')

REPO_URL = 'https://github.com/syedsafiullah777/CAPSTONE--RAG-WITH-UNCERTAINITY-QUANTIFICATION-.git'
BRANCH = 'main'
CLONE_DIR = Path('/content/capstone-rag')

if CLONE_DIR.exists():
    !rm -rf {CLONE_DIR}

print('Cloning branch:', BRANCH)
result = subprocess.run(
    ['git', 'clone', '--depth', '1', '--branch', BRANCH, REPO_URL, str(CLONE_DIR)],
    capture_output=True,
    text=True,
)
if result.returncode != 0:
    print(result.stderr)
    raise RuntimeError(f'git clone failed. Push V2/ to GitHub on branch {BRANCH!r} first.')

V2_ROOT = CLONE_DIR / 'V2'
if not (V2_ROOT / 'scripts' / 'run_judge.py').is_file():
    raise FileNotFoundError(f'Judge script missing at {V2_ROOT}. Push Phase 16 judge code to GitHub first.')

os.chdir(V2_ROOT)
sys.path.insert(0, str(V2_ROOT))
os.environ['V2_REQUIRE_CUDA'] = '1'
os.environ['V2_FORBID_MOCK'] = '1'
print('OK — working in V2_ROOT:', V2_ROOT)
!git -C {CLONE_DIR} log -1 --oneline

## 2. Install dependencies

In [ ]:
!pip -q install -r requirements.txt
!pip -q install llama-cpp-python --extra-index-url https://abetlen.github.io/llama-cpp-python/whl/cu122

## 3. Mount Drive and copy frozen Phase 15 JSONL

No knowledge-base restore. No retrieval. No RAG rerun.

In [ ]:
from google.colab import drive
from pathlib import Path
import hashlib
import os
import shutil

drive.mount('/content/drive')

V2 = Path('/content/capstone-rag/V2')
DRIVE_ROOT = Path('/content/drive/MyDrive/MSc-RAG')
os.environ['V2_DRIVE_ROOT'] = str(DRIVE_ROOT)
EXPECTED = 'f5256ae40fa8db0d6172ff9f4083bbde6c1c4fdb47916baa73529bc8215caafa'
rel = 'results/raw/phase15_benchmark/phase15_20260826T203744Z_dae9c3a4/cases.jsonl'
src = DRIVE_ROOT / rel
dst = V2 / rel
if not src.is_file():
    raise FileNotFoundError(f'Frozen Phase 15 JSONL missing on Drive: {src}')
dst.parent.mkdir(parents=True, exist_ok=True)
shutil.copy2(src, dst)
digest = hashlib.sha256(dst.read_bytes()).hexdigest()
print('copied', dst)
print('sha256', digest)
if digest != EXPECTED:
    raise RuntimeError('Phase 15 JSONL SHA-256 mismatch. Refusing to judge.')
print('SHA-256 OK — frozen Phase 15 input accepted')

## 4. SHA + lock check (do not recalibrate T; do not rewrite Phase 15)

In [ ]:
from src.calibration.lock import load_official_lock
from src.evaluation.runner import EXPECTED_RAW_SHA256, sha256_file
from pathlib import Path

raw = Path('results/raw/phase15_benchmark/phase15_20260826T203744Z_dae9c3a4/cases.jsonl')
digest = sha256_file(raw)
print('source sha256', digest)
if digest != EXPECTED_RAW_SHA256:
    raise RuntimeError('SHA mismatch. Do not judge a rewritten JSONL.')
lock = load_official_lock()
print('locked T', lock['threshold'], 'used_frozen_test_140', lock.get('used_frozen_test_140'))
if float(lock['threshold']) != 0.65:
    raise RuntimeError('Expected locked T=0.65. Do not recalibrate.')
print('Judge will use saved answers + retrieved evidence only. No RAG rerun.')

## 5. Official 420-case judge (`llama_cpp`, one Qwen3-8B instance)

Incremental `judge.jsonl` + Drive checkpoint after each case. If interrupted, use the **resume** cell.

In [ ]:
import os
os.environ['V2_REQUIRE_CUDA'] = '1'
os.environ['V2_FORBID_MOCK'] = '1'
os.environ['V2_DRIVE_ROOT'] = '/content/drive/MyDrive/MSc-RAG'
!PYTHONPATH=. python scripts/run_judge.py --backend llama_cpp

## 5b. Resume after disconnect (only if section 5 did not finish)

Skips completed `{architecture}:{question_id}` keys. Retries failed cases. Does not restart from case 1. Does not rerun RAG.

In [ ]:
# Uncomment only after an interrupted 420-case judge run:
# import os
# os.environ['V2_REQUIRE_CUDA'] = '1'
# os.environ['V2_DRIVE_ROOT'] = '/content/drive/MyDrive/MSc-RAG'
# !PYTHONPATH=. python scripts/run_judge.py --backend llama_cpp --resume-latest

## 6. Completion summary — 420 judge cases, no RAG rerun

In [ ]:
import json
from pathlib import Path
from src.evaluation.runner import EXPECTED_RAW_SHA256, sha256_file

summary = Path('results/config/phase16_judge_summary.json')
smoke = Path('results/config/phase16_judge_smoke_test.json')
print('summary:', summary.is_file(), 'record:', smoke.is_file())
data = json.loads(summary.read_text())
print('status:', data.get('status'))
print('run_id:', data.get('run_id'))
print('metric:', data.get('metric_label'))
print('completed/failed/pending:', data.get('n_completed'), data.get('n_failed'), data.get('n_pending'))
print('parse_failures:', data.get('n_parse_failure'))
print('used_rag_rerun:', data.get('used_rag_rerun'))
print('source_sha:', data.get('source_raw_sha256'))

if data.get('used_rag_rerun') is not False:
    raise RuntimeError('Judge must not rerun RAG.')
if data.get('source_raw_sha256') != EXPECTED_RAW_SHA256:
    raise RuntimeError('Source SHA mismatch')
raw15 = Path('results/raw/phase15_benchmark/phase15_20260826T203744Z_dae9c3a4/cases.jsonl')
if sha256_file(raw15) != EXPECTED_RAW_SHA256:
    raise RuntimeError('Phase 15 JSONL changed during judging')
if data.get('device') == 'mps_capable_host':
    raise RuntimeError('This is a Mac result, not Colab GPU.')
if data.get('backend') != 'llama_cpp':
    raise RuntimeError(f'Expected llama_cpp, got {data.get("backend")}')
if int(data.get('n_planned') or 0) != 420:
    raise RuntimeError('Official judge must plan 420 cases.')

judge_raw = Path(data['raw_path'])
if not judge_raw.is_file():
    judge_raw = Path('/content/capstone-rag/V2') / data['raw_path']
rows = [json.loads(line) for line in judge_raw.read_text().splitlines() if line.strip()]
last = {}
for row in rows:
    last[row.get('case_key')] = row
print('judge_lines:', len(rows), 'unique_keys:', len(last))
if data.get('status') == 'PASS' and len(last) != 420:
    raise RuntimeError(f'PASS requires 420 unique keys, got {len(last)}')
print('Not official RAGAS. CPU Phase 16 metrics were not rewritten.')

## 7. Copy judge results to Google Drive

Target: `MyDrive/MSc-RAG/results/raw/phase16_judge/`

In [ ]:
from google.colab import drive
from pathlib import Path
import json
import shutil

drive.mount('/content/drive', force_remount=True)
V2 = Path('/content/capstone-rag/V2')
DRIVE = Path('/content/drive/MyDrive/MSc-RAG')

summary = json.loads((V2 / 'results' / 'config' / 'phase16_judge_summary.json').read_text())
run_id = summary['run_id']

raw_src = V2 / 'results' / 'raw' / 'phase16_judge' / run_id
raw_dest = DRIVE / 'results' / 'raw' / 'phase16_judge' / run_id
raw_dest.parent.mkdir(parents=True, exist_ok=True)
if raw_src.is_dir():
    shutil.copytree(raw_src, raw_dest, dirs_exist_ok=True)
    print('copied raw', raw_dest)

ckpt_src = V2 / 'results' / 'checkpoints' / 'phase16_judge'
ckpt_dest = DRIVE / 'checkpoints' / 'phase16_judge'
if ckpt_src.is_dir():
    shutil.copytree(ckpt_src, ckpt_dest, dirs_exist_ok=True)
    print('copied checkpoints', ckpt_dest)

metrics_src = V2 / 'results' / 'metrics'
metrics_dest = DRIVE / 'results' / 'metrics'
metrics_dest.mkdir(parents=True, exist_ok=True)
for name in ('phase16_judge_summary.csv', 'phase16_judge_summary.md', 'phase16_judge_by_architecture.json'):
    src = metrics_src / name
    if src.is_file():
        shutil.copy2(src, metrics_dest / name)
        print('copied', name)

cfg_dest = DRIVE / 'configs' / 'phase16'
cfg_dest.mkdir(parents=True, exist_ok=True)
for name in (
    'phase16_judge_runtime_fingerprint.json',
    'phase16_judge_smoke_test.json',
    'phase16_judge_summary.json',
):
    src = V2 / 'results' / 'config' / name
    if src.is_file():
        shutil.copy2(src, cfg_dest / name)
        print('copied', name)
print('Drive dest root:', raw_dest)
print('final status', summary.get('status'), 'completed', summary.get('n_completed'), '/', summary.get('n_planned'))